<a href="https://colab.research.google.com/github/Hania-Emaan/code-switching-codesaviours-si26-Hania-Emaan/blob/main/SI26_Week7_Hania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Cell 1 — Load dataset and prepare for training
!pip install transformers torch datasets seqeval

import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Load dataset (checks local first, then Google Drive)
if os.path.exists('dataset.csv'):
    file_path = 'dataset.csv'
else:
    file_path = '/content/drive/MyDrive/CodeSaviours_Project2/dataset.csv'

df = pd.read_csv(file_path)

# Create label mapping
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

# Group by sentence
sentence_col = 'sentence' if 'sentence' in df.columns else 'sentence_id'
word_col = 'word' if 'word' in df.columns else 'tokens'
label_col = 'label' if 'label' in df.columns else 'labels'

sentences = []
for _, group in df.groupby(sentence_col):
    sentences.append({
        'words': group[word_col].astype(str).tolist(),
        'labels': group[label_col].tolist()
    })

# Split into train (80%) and test (20%) with stratify check
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

print(f'Total sentences: {len(sentences)}')
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

Total sentences: 80
Training sentences: 64
Testing sentences: 16


In [12]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from seqeval.metrics import classification_report
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                        TrainingArguments, Trainer, DataCollatorForTokenClassification)

# 1. Load Dataset (Checks local file first, then Google Drive)
if os.path.exists('dataset.csv'):
    file_path = 'dataset.csv'
else:
    file_path = '/content/drive/MyDrive/CodeSaviours_Project2/dataset.csv'

df = pd.read_csv(file_path)

label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

sentence_col = 'sentence' if 'sentence' in df.columns else 'sentence_id'
word_col = 'word' if 'word' in df.columns else 'tokens'
label_col = 'label' if 'label' in df.columns else 'labels'

# Safely extract sentences and words as strings
sentences = []
for _, group in df.groupby(sentence_col):
    sentences.append({
        'words': [str(w) for w in group[word_col].tolist()],
        'labels': group[label_col].tolist()
    })

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

# 2. Tokenizer & Alignment
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_dataset = Dataset.from_dict({
    "words": [item["words"] for item in train_data],
    "labels": [item["labels"] for item in train_data]
}).map(tokenize_and_align_labels, batched=True)

test_dataset = Dataset.from_dict({
    "words": [item["words"] for item in test_data],
    "labels": [item["labels"] for item in test_data]
}).map(tokenize_and_align_labels, batched=True)

# 3. Model & Trainer Configuration
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=12,              # Increased to 12 for learning MIX patterns
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=3e-5,               # Slightly higher learning rate for small datasets
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

# 4. Train
print("Starting training...")
trainer.train()
print("Training complete!")

# 5. Evaluate and print F1 scores
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

print("\n--- EVALUATION REPORT ---")
report = classification_report(true_labels, true_predictions)
print(report)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!



--- EVALUATION REPORT ---
              precision    recall  f1-score   support

          IX       1.00      0.62      0.76        13
          NG       0.93      0.94      0.93        83
          RD       0.62      0.62      0.62        13

   micro avg       0.90      0.86      0.88       109
   macro avg       0.85      0.72      0.77       109
weighted avg       0.90      0.86      0.88       109



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [17]:
from transformers import pipeline

# Load pipeline using your local model and tokenizer
# Added ignore_labels=[] so it prints URD, ENG, and MIX without skipping any
nlp_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    ignore_labels=[]
)

# Test sentence
text = "you don't listen me. me tmse bohat tang hun"

# Run inference
predictions = nlp_pipeline(text)

print("\n--- Predictions ---")
for pred in predictions:
    print(f"Entity: {pred['entity_group']} | Text: '{pred['word']}' | Score: {pred['score']:.4f}")


--- Predictions ---
Entity: ENG | Text: 'you don't listen me. me t' | Score: 0.8332
Entity: URD | Text: 'mse bohat' | Score: 0.8023
Entity: ENG | Text: 'tang' | Score: 0.5323
Entity: URD | Text: 'hun' | Score: 0.5684


In [15]:
# Define repository name
repo_name = 'code-switching-codesaviours-si26-hania'

# Push updated model and tokenizer
print("Pushing updated model to Hugging Face...")
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("Published successfully! Updated model is live on Hugging Face.")

Pushing updated model to Hugging Face...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sndr0xi/model.safetensors:   1%|          | 9.79MB / 1.11GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp42nugkdc/tokenizer.json:  93%|#########3| 15.9MB / 17.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Published successfully! Updated model is live on Hugging Face.


In [20]:
import torch
from datasets import load_dataset # Or your local test dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
import numpy as np
import evaluate

# 1. Load your fine-tuned model and tokenizer
MODEL_PATH = "HaniaEmaan/code-switching-codesaviours-si26-hania" # Replace with your local folder path if testing locally

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

# 2. Define label mappings (ensure these match your training setup)
id2label = model.config.id2label
label2id = model.config.label2id
label_list = list(id2label.values())

# 3. Load metric evaluator (seqeval is standard for token classification)
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert IDs back to string labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute sequence evaluation metrics
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)

    # Calculate global token-level accuracy manually for clear visibility
    flat_preds = [p for sublist in true_predictions for p in sublist]
    flat_labels = [l for sublist in true_labels for l in sublist]
    correct = sum(1 for p, l in zip(flat_preds, flat_labels) if p == l)
    total = len(flat_labels)
    overall_accuracy = correct / total if total > 0 else 0.0

    return {
        "overall_accuracy": overall_accuracy,
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

# 4. Run Evaluation using Hugging Face Trainer
# Note: Ensure 'test_dataset' is pre-processed/tokenized with ground truth labels
training_args = TrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset, # Pass your tokenized test dataset here
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("--- Running Evaluation on Test Set ---")
eval_results = trainer.evaluate()

# Print formatted final results
print("\n=== FINAL MODEL PERFORMANCE ===")
print(f"Token-Level Accuracy : {eval_results['eval_overall_accuracy'] * 100:.2f}%")
print(f"Overall Precision    : {eval_results['eval_precision']:.4f}")
print(f"Overall Recall       : {eval_results['eval_recall']:.4f}")
print(f"Overall Macro F1     : {eval_results['eval_f1']:.4f}")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Running Evaluation on Test Set ---


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Training Loss,Validation Loss,Step,Overall Accuracy,Precision,Recall,F1
No log,0.272143,0,0.908333,0.895238,0.862385,0.878505



=== FINAL MODEL PERFORMANCE ===
Token-Level Accuracy : 90.83%
Overall Precision    : 0.8952
Overall Recall       : 0.8624
Overall Macro F1     : 0.8785


In [19]:
!pip install evaluate seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
